# GP - SWIM Experiments

In [1]:
import torch
import gpytorch
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

## STAGE 1: Create a TOY dataset and fit an Exact Gaussian Process 

In [2]:
torch.manual_seed(42)

# ─── 1. Create dataset ───────────────────────────────────
N_train = 100
N_test  = 300

# Input: uniform in [-3, 3]
X_train = torch.linspace(-3, 3, N_train).unsqueeze(1)  # shape (100, 1)
y_train = torch.sin(X_train.squeeze()) + 0.1 * torch.randn(N_train)

X_test  = torch.linspace(-4, 4, N_test).unsqueeze(1)   # shape (300, 1)
y_test  = torch.sin(X_test.squeeze())                   # noiseless ground truth

print(f"X_train: {X_train.shape},  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},   y_test:  {y_test.shape}")


X_train: torch.Size([100, 1]),  y_train: torch.Size([100])
X_test:  torch.Size([300, 1]),   y_test:  torch.Size([300])


In [10]:
# ─── 2. Define GP model ──────────────────────────────────
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, X_train, y_train, likelihood):
        super().__init__(X_train, y_train, likelihood)
        self.mean_module  = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=X_train.shape[1])
        )

    def forward(self, x):
        mean  = self.mean_module(x)
        covar = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean, covar) # type: ignore

In [11]:
# ─── 3. Initialize ───────────────────────────────────────
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model      = ExactGPModel(X_train, y_train, likelihood)

In [12]:
# ─── 4. Train ────────────────────────────────────────────
model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll       = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

num_iters = 100 
for i in range(num_iters):
    optimizer.zero_grad()
    loss = -mll(model(X_train), y_train) # type: ignore
    loss.backward()
    optimizer.step()

print(f"\nGP fitted successfully.")
print(f"  Length scale: {model.covar_module.base_kernel.lengthscale.item():.4f}")
print(f"  Output scale: {model.covar_module.outputscale.item():.4f}")
print(f"  Noise:        {likelihood.noise.item():.4f}")
print(f"  Mean const:   {model.mean_module.constant.item():.4f}") # type: ignore


GP fitted successfully.
  Length scale: 1.2596
  Output scale: 0.6832
  Noise:        0.0089
  Mean const:   0.1524


In [6]:
# ─── 5. Freeze GP ────────────────────────────────────────
model.eval()
likelihood.eval()
print(f"\nGP frozen. Ready for Stage 2 — pair sampling.")


GP frozen. Ready for Stage 2 — pair sampling.


In [7]:
import sys
sys.path.insert(0, '/Users/gizemnurdal/Workspace/swim-meets-kans')

import importlib
from sgkan.gaussian_process_models import (
    init_gp,
    train_gp,
    predict
)

In [8]:
model, likelihood = init_gp(X_train, y_train)

[DEBUG] Initialized ExactGPModel: X_train.shape=torch.Size([100, 1])


In [9]:
model, likelihood = train_gp(model, likelihood, X_train, y_train)

[DEBUG] ExactGPModel and likelihood unfrozen (train mode)
[DEBUG] Training ExactGPModel: 100 iters, lr=0.1, loss=MLL
  Iter 20/100, Loss: -0.598483
  Iter 40/100, Loss: -0.731419
  Iter 60/100, Loss: -0.731950
  Iter 80/100, Loss: -0.732681
  Iter 100/100, Loss: -0.732744
[DEBUG] Training complete.
  Lengthscale: 1.2557
  Outputscale: 0.7124
  Noise: 0.008845
  Mean const: 0.1523
[DEBUG] ExactGPModel and likelihood frozen (eval mode)


/Users/gizemnurdal/miniconda3/envs/swim-meets-kans/lib/python3.11/site-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


## STAGE 2: GP Driven SWIM Scores

In [ ]:
"""
    X_train,          # torch tensor (N, d)
    y_train,          # torch tensor (N,)
    model,            # fitted frozen GP model
    likelihood,       # fitted frozen likelihood
    M,                # number of candidate pairs
    N_pairs,          # number of pairs to select (= layer_width equivalent)
    T=3,              # number of interior points per pair
    epsilon=1e-8,     # numerical stability
    random_seed=42
"""

random_seed = 42
rng = np.random.default_rng(random_seed)
N = X_train.shape[0]

In [ ]:
# ── Step 1: Sample M candidate pairs ─────────────────
# Same logic as SWIM — delta trick guarantees idx_from != idx_to
M = 100 # Update later
idx_from = rng.integers(low=0, high=N, size=M)
delta    = rng.integers(low=1, high=N-1, size=M)
idx_to   = (idx_from + delta) % N

# Select corr. values using the indices list
x_a = X_train[idx_from]   # shape (M, d)
x_b = X_train[idx_to]     # shape (M, d)
y_a = y_train[idx_from]   # shape (M,)
y_b = y_train[idx_to]   # shape (M,)

In [ ]:
# ── Step 2: Create T interior points per pair ────────
# t in {1/(T+1), 2/(T+1), ..., T/(T+1)} — avoids endpoints
T = 3
t_values = torch.linspace(0, 1, T+2)[1:-1]  # shape (T,)
# x_t shape: (M, T, d)
# x_a[:, None, :] broadcasts to (M, 1, d)
x_interior = (
    x_a.unsqueeze(1) +
    t_values.view(1, T, 1) * (x_b - x_a).unsqueeze(1)
)  # (M, T, d)
# Flatten to (M*T, d) for single GP query
x_interior_flat = x_interior.reshape(M * T, -1)

In [ ]:
# ── Step 3: Query frozen GP at interior points ───────
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred         = likelihood(model(x_interior_flat))
    mu_interior  = pred.mean.reshape(M, T)      # (M, T)
    std_interior = pred.variance.sqrt().reshape(M, T)  # (M, T)

# ── Endpoint gradients: need grad through mu ──
x_a_g = x_a.detach().requires_grad_(True)  # (M, d)
x_b_g = x_b.detach().requires_grad_(True)  # (M, d)

with gpytorch.settings.fast_pred_var():
    pred_a = likelihood(model(x_a_g))
    pred_b = likelihood(model(x_b_g))
    
    mu_a  = pred_a.mean          # (M,)
    std_a = pred_a.variance.sqrt()  # (M,)
    
    mu_b  = pred_b.mean          # (M,)
    std_b = pred_b.variance.sqrt()  # (M,)

# ── Numerator: L-inf norm of gradient difference ──
grad_a = torch.autograd.grad(mu_a.sum(), x_a_g)[0]  # (M, d)
grad_b = torch.autograd.grad(mu_b.sum(), x_b_g)[0]  # (M, d)

numerator = (grad_a - grad_b).abs().max(dim=1).values  # (M,)

# ── Denominator: uncertainty at endpoints + along segment ──
epsilon = 1e-6
denominator = std_a + std_interior.sum(dim=1) + std_b + epsilon  # (M,) # dont add boundaries

# ── Scores and probabilities ──
scores      = numerator / denominator          # (M,)
probs       = scores / scores.sum()            # (M,)  sums to 1

In [ ]:
scores

In [ ]:
probs

In [ ]:
probs.sum()

In [ ]:
# ── Step 5: Sample winning pairs ──
layer_width = 10
probs_np = probs.detach().cpu().numpy()  # multinomial needs numpy for rng.choice

selected_idx = rng.choice(
    M,                        # sample from M candidates
    size=layer_width,         # pick layer_width winners
    replace=True,             # same pair can be selected multiple times
    p=probs_np
)

# Index into your pair tensors
x_a_selected = x_a[selected_idx]  # (layer_width, d)
x_b_selected = x_b[selected_idx]  # (layer_width, d)

In [ ]:
x_a_selected

In [ ]:
x_b_selected

In [ ]:
# ── Step 7: Sample GP posterior functions over selected segments ──

# Create dense interior points for each selected pair (for smooth function)
T_sample = 200  # more points for a smooth curve equivalent to 50
t_dense  = torch.linspace(0, 1, T_sample)  # (T_sample,)

# Interior points for selected pairs only
x_segments = (
    x_a_selected.unsqueeze(1) +
    t_dense.view(1, T_sample, 1) * (x_b_selected - x_a_selected).unsqueeze(1)
)  # (layer_width, T_sample, d)

# Flatten for GP query
x_segments_flat = x_segments.reshape(layer_width * T_sample, -1)  # (layer_width*T_sample, d)

# Get posterior distribution over these points
with gpytorch.settings.fast_pred_var():
    pred_segments = likelihood(model(x_segments_flat))

# Reshape mean and covariance for sampling
# We need to sample per segment separately
sampled_functions = []

for i in range(layer_width):
    # Points for this segment
    x_seg_i = x_segments[i]  # (T_sample, d)
    
    with gpytorch.settings.fast_pred_var():
        pred_i = likelihood(model(x_seg_i))
    
    # Sample one function from the posterior
    f_sample = pred_i.mean  # (T_sample,)
    sampled_functions.append(f_sample)

sampled_functions = torch.stack(sampled_functions)  # (layer_width, T_sample)

In [ ]:
x_segments.shape, x_segments[0]

In [ ]:
sampled_functions[0]

In [ ]:
sampled_functions

In [ ]:
sampled_functions.shape

In [ ]:
# ── Step 8: Interpolate edge functions at X_train and X_test ──
H_train = torch.zeros(N_train, layer_width)
H_test  = torch.zeros(N_test,  layer_width)

for i in range(layer_width):
    seg_x = x_segments[i, :, 0].detach().numpy()   # (T_sample,) — x positions of edge i
    seg_f = sampled_functions[i].detach().numpy()   # (T_sample,) — φi values at those x positions

    x_train_np = X_train[:, 0].detach().numpy()    # (N_train,)
    x_test_np  = X_test[:, 0].detach().numpy()     # (N_test,)

    # For each training point: interpolate φi(x) from the lookup table
    H_train[:, i] = torch.tensor(np.interp(x_train_np, seg_x, seg_f))
    H_test[:, i]  = torch.tensor(np.interp(x_test_np,  seg_x, seg_f))

In [ ]:
layer_width

In [ ]:
x_train_np

In [ ]:
H_train

In [ ]:
H_train.shape

In [ ]:
# ── Step 9: OLS — solve for output layer ──
H_train_b = torch.cat([H_train, torch.ones(N_train, 1)], dim=1)  # (N_train, layer_width+1)
H_test_b  = torch.cat([H_test,  torch.ones(N_test,  1)], dim=1)  # (N_test,  layer_width+1)

result = torch.linalg.lstsq(H_train_b, y_train.unsqueeze(1))
W_out  = result.solution  # (layer_width+1, 1)

# ── Step 10: Predict and evaluate ──
y_pred = H_test_b @ W_out
mse    = ((y_pred.squeeze() - y_test) ** 2).mean()
print(f"\nTest MSE: {mse.item():.6f}")

In [ ]:
# Baseline 1: GP posterior mean directly
with torch.no_grad():
    gp_pred = likelihood(model(X_test)).mean
    gp_mse  = ((gp_pred - y_test) ** 2).mean()
    print(f"GP baseline MSE:   {gp_mse.item():.6f}")

# Baseline 2: predicting mean of y_train
# double check
mean_pred = y_train.mean().expand(N_test)
mean_mse  = ((mean_pred - y_test) ** 2).mean()
print(f"Mean baseline MSE: {mean_mse.item():.6f}")

In [ ]:
# Relative L2 error = ||y_pred - y_test||_2 / ||y_test||_2
rel_l2 = torch.norm(y_pred.squeeze() - y_test) / torch.norm(y_test)
print(f"Relative L2 error: {rel_l2.item():.6f}")

# For all baselines too
gp_rel_l2   = torch.norm(gp_pred - y_test) / torch.norm(y_test)
mean_rel_l2 = torch.norm(mean_pred - y_test) / torch.norm(y_test)

print(f"GP baseline relative L2:   {gp_rel_l2.item():.6f}")
print(f"Mean baseline relative L2: {mean_rel_l2.item():.6f}")

In [ ]:
# take different function types
# describe and write in your thesis